In [ ]:
pip install fsspec pandera numpy matplotlib pandas numpy pandas pyarrow aiohttp requests

In [ ]:
pip install pyarrow.parquet

In [ ]:
import fsspec
print(fsspec.__version__)

In [3]:
import numpy as np
import pandas as pd
import pyarrow as pa_arrow
import pyarrow.parquet as pq

In [ ]:
url = 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2020_csv.zip'

#Definindo variáveis q serão utilizadas durante a execução do código.
arquivo_local = None
url_natalidade = url
n_amostra = 500_000

fonte = arquivo_local if (arquivo_local and Path(arquivo_local).exists()) else url_natalidade
print(f'Fonte configurada: {fonte[:80]}...' if len(fonte)>80 else f'Fonte: {fonte}')

In [ ]:
import requests
import zipfile
import io

print(f'{"_"*5}')
print("Baixando e descompactando o arquivo em memória")
print()
print(f'{"_"*50}')

In [ ]:
# Download do arquivo bruto - conteúdo binário

resposta_da_requisicao = requests.get(url_natalidade)
conteudo_do_arquivo = io.BytesIO(resposta_da_requisicao.content)

### 1. `resposta_da_requisicao = requests.get(url_natalidade)`

O `requests.get` é como se você estivesse digitando um endereço no seu navegador e apertando "Enter".
* **O que faz:** Envia uma requisição HTTP para a URL armazenada na variável `url_natalidade`.
* **O resultado:** Ele retorna um **objeto de resposta**. Esse objeto contém tudo o que veio do servidor: o conteúdo do arquivo, o código de status (se deu certo ou erro 404, por exemplo), os cabeçalhos, etc.

### 2. `conteudo_do_arquivo = io.BytesIO(resposta_da_requisicao.content)`

Esta linha é um "truque" de engenharia para tratar dados binários. Vamos dividir em duas partes:

* **`resposta_da_requisicao.content`**: Aqui você está acessando especificamente o "corpo" da resposta em formato de **bytes** (dados brutos). Se o arquivo for uma imagem ou um arquivo Excel, os dados estarão nesse formato binário.
* **`io.BytesIO(...)`**: O Python normalmente espera ler arquivos que estão gravados no disco (com um caminho tipo `C:/documentos/arquivo.csv`). O `BytesIO` cria um **arquivo virtual** na memória RAM.
* **O efeito prático:** Agora, a variável `conteudo_do_arquivo` se comporta exatamente como se fosse um arquivo aberto no seu computador.

---

### Por que usar isso?

Imagine que você está baixando uma planilha do IBGE. Em vez de fazer isso:
1. Baixar o arquivo.
2. Salvar em `C:/downloads/dados.xlsx`.
3. Abrir o arquivo com o Pandas.
4. Deletar o arquivo depois.

Você faz tudo direto na memória, o que é **mais rápido** e **mais limpo**, pois não deixa "lixo" (arquivos temporários) na sua máquina.

> **Dica:** Geralmente, após essas linhas, você usaria algo como `pandas.read_excel(conteudo_do_arquivo)` para começar a analisar os dados de natalidade.


In [ ]:
# "Des"compactação e leitura

with zipfile.ZipFile(conteudo_do_arquivo) as zippado:
    arquivos_internos = zippado.namelist()
    with zippado.open(arquivos_internos[0]) as f:
        df_natalidade2024 = pd.read_csv(f, sep=';', nrows=n_amostra, encoding='latin1')

### 1. `with zipfile.ZipFile(conteudo_do_arquivo) as zippado:`
* **O que faz:** O comando `ZipFile` lê os dados binários que você baixou (que estão na variável `conteudo_do_arquivo`) e os interpreta como um arquivo compactado .zip.
* **O `with`:** É uma boa prática em Python chamada "Gerenciador de Contexto". Ele garante que, assim que você terminar de ler o conteúdo, o arquivo seja "fechado" na memória, liberando espaço.

### 2. `arquivos_internos = zippado.namelist()`
* **O que faz:** Um arquivo ZIP pode ter vários arquivos dentro dele (planilhas, textos, fotos). O `namelist()` cria uma lista com os nomes de tudo o que está lá dentro.
* **Exemplo:** Se dentro do ZIP tiver um arquivo chamado `natalidade_2024.csv`, essa lista será `['natalidade_2024.csv']`.

### 3. `with zippado.open(arquivos_internos[0]) as f:`
* **O que faz:** Ele acessa o primeiro arquivo da lista (`[0]`) e o "abre" para leitura. A variável `f` passa a ser o fluxo de dados desse arquivo específico que estava escondido dentro do ZIP.

### 4. `df = pd.read_csv(f, sep=';', ...)`
Aqui é onde a mágica acontece e os dados viram uma tabela (DataFrame) do Pandas:
* **`f`**: O Pandas lê diretamente do arquivo aberto na memória.
* **`sep=';'`**: Avisa que as colunas no arquivo CSV são separadas por ponto e vírgula.
* **`nrows=n_amostra`**: Comando muito útil! Ele diz ao Python: "Não leia o arquivo inteiro (que pode ter milhões de linhas), leia apenas as primeiras `X` linhas". Isso economiza muita memória.
* **`encoding='latin1'`**: Define o padrão de caracteres. No Brasil, arquivos de órgãos governamentais costumam usar `latin1` para que acentos e cedilhas não fiquem bugados.

---

### Resumo do Fluxo


1. **`ZipFile`**: Identifica o "pacote".
2. **`namelist`**: Olha o que tem na "etiqueta" do pacote.
3. **`open`**: Abre um item específico do pacote.
4. **`read_csv`**: Transforma o texto desse item em uma tabela organizada.

In [ ]:
# Validação da Compressão

compressao = "ZIP" if fonte.endswith('.zip') else "Nenhuma"

In [ ]:
uso_memoria = df_natalidade2024.memory_usage(deep=True) / 1024

print(f'{"Coluna":<21}|{"Tipo":<12}|{"RAM":<5}')
print(f'{"_"*50}')
for meta in df_natalidade2024.columns:
    tipo = str(df_natalidade2024[meta].dtype)
    espaco = uso_memoria[meta]
    print(f' {meta:<20}| {tipo:<10} |{espaco:>12.1f}')
print(f'{"_"*50}')

In [ ]:
df_natalidade2024.head()

Aqui está a lista organizada por categorias para facilitar o seu entendimento:

### 1. Informações de Identificação e Registro
* **contador:** Um número sequencial para controle interno do arquivo.
* **ORIGEM:** Indica se o dado veio de uma base estadual, municipal ou federal.
* **CODESTAB:** Código do Cadastro Nacional de Estabelecimentos de Saúde (CNES) onde ocorreu o nascimento.
* **DTCADASTRO / DTRECEBIM / DTRECORIGA:** Datas de cadastro, recebimento e registro original no sistema.
* **NUMEROLOTE:** Número do lote de processamento dos dados.
* **VERSAOSIST:** Versão do sistema de software utilizada para coletar os dados.

### 2. Localização Geográfica
* **CODMUNNASC:** Código do município onde a criança nasceu (padrão IBGE).
* **LOCNASC:** Tipo de local do nascimento (1 para Hospital, 2 para outro estabelecimento de saúde, 3 para Domicílio, etc.).
* **CODMUNRES:** Código do município onde a mãe reside.
* **CODMUNNATU / CODUFNATU:** Município e Unidade da Federação de naturalidade da mãe.
* **CODPAISRES:** Código do país de residência (geralmente Brasil).

### 3. Características da Mãe e do Pai
* **IDADEMAE / IDADEPAI:** Idade da mãe e do pai em anos.
* **ESTCIVMAE:** Estado civil da mãe (Solteira, casada, viúva, etc.).
* **ESCMAE / ESCMAE2010 / ESCMAEAGR1:** Escolaridade da mãe (existem diferentes colunas por conta de mudanças metodológicas ao longo dos anos).
* **SERIESCMAE:** Série escolar que a mãe cursava/concluiu.
* **CODOCUPMAE:** Código da ocupação da mãe (baseado na CBO - Classificação Brasileira de Ocupações).
* **RACACORMAE:** Raça ou cor da mãe.
* **DTNASCMAE:** Data de nascimento da mãe.
* **NATURALMAE:** Nacionalidade da mãe.

### 4. Histórico Reprodutivo (Paridade)
* **QTDFILVIVO / QTDFILMORT:** Quantidade de filhos vivos e mortos em gestações anteriores.
* **QTDGESTANT:** Número total de gestações anteriores.
* **QTDPARTNOR / QTDPARTCES:** Quantidade de partos normais e cesáreas anteriores.
* **PARIDADE:** Indica se a mãe é primípara (primeiro filho) ou multipara.

### 5. Detalhes da Gestação e Parto
* **GESTACAO:** Semanas de gestação (em faixas, ex: 37 a 41 semanas).
* **SEMAGESTAC:** Número exato de semanas de gestação.
* **TPMETESTIM:** Método utilizado para estimar a idade gestacional (ex: Exame físico ou Ultrassom).
* **GRAVIDEZ:** Tipo de gravidez (Única, dupla, tripla ou mais).
* **PARTO:** Tipo de parto (Vaginal ou Cesáreo).
* **CONSULTAS / CONSPRENAT:** Número de consultas de pré-natal realizadas.
* **MESPRENAT:** Em qual mês de gestação a mãe iniciou o pré-natal.
* **DTULTMENST:** Data da última menstruação.
* **STTRABPART:** Indica se a mãe entrou em trabalho de parto.
* **STCESPARTO:** Indica se a cesárea ocorreu antes do trabalho de parto iniciar.
* **TPAPRESENT:** Tipo de apresentação do bebê (Cefálica, pélvica, etc.).

### 6. Características do Recém-Nascido
* **DTNASC / HORANASC:** Data e hora exata do nascimento.
* **SEXO:** Sexo do bebê (Masculino ou Feminino).
* **APGAR1 / APGAR5:** Notas de Apgar no 1º e 5º minuto (avalia a vitalidade do bebê).
* **RACACOR:** Raça ou cor do recém-nascido.
* **PESO:** Peso ao nascer em gramas.
* **IDANOMAL / CODANOMAL:** Indica se foi detectada alguma anomalia congênita e qual o código (CID-10).

### 7. Indicadores e Classificações Técnicas
* **TPROBSON:** Classificação de Robson (agrupa as mulheres em 10 grupos para analisar taxas de cesárea).
* **KOTELCHUCK:** Índice que mede a adequação do pré-natal (combina o início do pré-natal com o número de consultas).
* **OPORT_DN:** Indicador de oportunidade de registro do nascimento.
* **DIFDATA:** Diferença de dias entre o nascimento e o registro.
* **TPNASCASSI:** Tipo de assistência no nascimento (Médico, enfermeira, parteira, etc.).
* **TPFUNCRESP / TPDOCRESP:** Tipo de função e documento do responsável pelo preenchimento da Declaração de Nascido Vivo.

---

**Uma dica rápida:** Se você estiver analisando esses dados em Python ou R, note que muitas colunas estão como `float64`. Isso geralmente acontece porque existem valores faltantes (`NaN`) na base, o que faz com que o sistema entenda os números inteiros como decimais.

In [ ]:
lista_colunas_principais = [
    #PAIS
    'IDADEMAE',
    'IDADEPAI',

    #GESTAÇÃO
    'SEMAGESTAC', #SEMANAS DE GESTAÇÃO
    'QTDPARTNOR', #QUANTIDADE DE PARTOS NORMAIS
    'QTDPARTCES', #QUANTIDADE DE PARTOS CESÁRIOS
    'CONSPRENAT', #QUANTIDADE DE PRENATAIS FEITOS.
    'KOTELCHUCK', #INDICE QUE CLASSIFICA A QUALIDADE DO PRENATAL FEITO
    'DIFDATA', #Diferença de dias entre o nascimento e o registro.
    'GRAVIDEZ', #INDICA SE A GRAVIDEZ FOI SIMPLES, DUPLA, TRIPLA E ETC.

    #BEBÊ E NASCIMENTO
    'PESO',
    'HORANASC',
    'DTNASC',
    'SEXO'
]

In [ ]:
type(lista_colunas_principais)

In [ ]:
#organizando um novo dataframe

df_final = df_natalidade2024[lista_colunas_principais].copy()

In [ ]:
df_final.head()

In [ ]:
df_final.info(memory_usage=True)

In [ ]:
df_final.to_parquet('sinasc2020_reduzidissima_final.parquet', index=False)

# Juntando todas os dfs

In [1]:
path = '/home/aluno/Downloads/git/'
import pandas as pd

In [2]:
df2015 = pd.read_parquet(path + 'sinasc2015_reduzidissima_final.parquet')
df2016 = pd.read_parquet(path + 'sinasc2016_reduzidissima_final.parquet')
df2017 = pd.read_parquet(path + 'sinasc2017_reduzidissima_final.parquet')
df2018 = pd.read_parquet(path + 'sinasc2018_reduzidissima_final.parquet')
df2019 = pd.read_parquet(path + 'sinasc2019_reduzidissima_final.parquet')
df2020 = pd.read_parquet(path + 'sinasc2020_reduzidissima_final.parquet')
df2021 = pd.read_parquet(path + 'sinasc2021_reduzidissima_final.parquet')
df2022 = pd.read_parquet(path + 'sinasc2022_reduzidissima_final.parquet')
df2023 = pd.read_parquet(path + 'sinasc2023_reduzidissima_final.parquet')
df2024 = pd.read_parquet(path + 'sinasc2024_reduzidissima_final.parquet')

In [3]:
df2016.columns = df2016.columns.str.upper()

In [4]:
df2016.head()

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,29.0,NaN,40.0,0.0,0.0,6.0,4,29,1.0,3985.0,340.0,18022016,2
1,14.0,NaN,NaN,0.0,0.0,4.0,3,50,1.0,2560.0,2140.0,28012016,2
2,30.0,NaN,37.0,0.0,1.0,10.0,5,47,1.0,3670.0,1045.0,25022016,2
3,23.0,NaN,36.0,0.0,1.0,6.0,4,61,1.0,2495.0,1330.0,20042016,2
4,30.0,39.0,38.0,0.0,1.0,4.0,3,54,1.0,2945.0,324.0,27042016,2


In [5]:
df2015.head()

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,31.0,NaN,NaN,NaN,NaN,NaN,9,50,1.0,2900.0,1730.0,8012015,2
1,28.0,NaN,NaN,NaN,NaN,NaN,9,147,1.0,4150.0,815.0,3022015,2
2,26.0,NaN,NaN,NaN,NaN,NaN,9,138,1.0,3490.0,800.0,12022015,2
3,21.0,NaN,NaN,NaN,NaN,NaN,9,29,1.0,3200.0,835.0,31032015,1
4,18.0,NaN,NaN,NaN,NaN,NaN,9,15,1.0,3200.0,2119.0,14042015,1


In [6]:
df2018.head()

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,28.0,35.0,39.0,0.0,1.0,8.0,5,38,1.0,3050.0,842.0,5032018,2
1,39.0,32.0,37.0,1.0,0.0,9.0,5,53,1.0,3440.0,1657.0,11012018,1
2,33.0,NaN,38.0,0.0,1.0,8.0,5,61,1.0,2920.0,1415.0,8022018,1
3,35.0,NaN,39.0,1.0,1.0,6.0,2,60,1.0,3020.0,1850.0,9022018,1
4,21.0,NaN,38.0,2.0,0.0,6.0,2,39,1.0,3785.0,336.0,2032018,2


In [7]:
df_final = pd.concat([df2015, df2016, df2017, df2018, df2019, df2020, df2021, df2022, df2023, df2024], axis=0, ignore_index=True)

In [8]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 13 columns):
 #   Column      Dtype  
---  ------      -----  
 0   IDADEMAE    float64
 1   IDADEPAI    float64
 2   SEMAGESTAC  float64
 3   QTDPARTNOR  float64
 4   QTDPARTCES  float64
 5   CONSPRENAT  float64
 6   KOTELCHUCK  int64  
 7   DIFDATA     int64  
 8   GRAVIDEZ    float64
 9   PESO        float64
 10  HORANASC    float64
 11  DTNASC      int64  
 12  SEXO        int64  
dtypes: float64(9), int64(4)
memory usage: 495.9 MB


In [9]:
df2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDADEMAE    499982 non-null  float64
 1   IDADEPAI    94705 non-null   float64
 2   SEMAGESTAC  489709 non-null  float64
 3   QTDPARTNOR  456259 non-null  float64
 4   QTDPARTCES  451175 non-null  float64
 5   CONSPRENAT  484848 non-null  float64
 6   KOTELCHUCK  500000 non-null  int64  
 7   DIFDATA     500000 non-null  int64  
 8   GRAVIDEZ    499583 non-null  float64
 9   PESO        499626 non-null  float64
 10  HORANASC    499467 non-null  float64
 11  DTNASC      500000 non-null  int64  
 12  SEXO        500000 non-null  int64  
dtypes: float64(9), int64(4)
memory usage: 49.6 MB
